# Research Module 1: Exploring AI in Radiology Data

**AI-Ready Radiology Curriculum**

In this notebook, you will:
1. Learn basic Python and pandas for data analysis
2. Load and explore a dataset of radiology AI findings
3. Visualize how AI performs across different imaging modalities
4. Calculate AI accuracy metrics used in radiology research
5. Complete a hands-on analysis task and commit your work to GitHub

---

**No prior programming experience is required.** Each code cell includes explanations. Run cells in order by clicking the play button or pressing `Shift + Enter`.

## Setup

First, we import the Python libraries we will use:
- **pandas**: for loading and analyzing tabular data (like a spreadsheet)
- **matplotlib**: for creating charts and visualizations

These libraries come pre-installed in Google Colab.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

print('Libraries loaded successfully!')
print(f'pandas version: {pd.__version__}')

## Step 1: Load the Dataset

We have a dataset of 60 radiology studies where an AI system flagged (or did not flag) findings, and a radiologist independently confirmed whether those findings were real.

Each row represents one study with these columns:
| Column | Description |
|--------|-------------|
| `study_id` | Anonymized study identifier |
| `modality` | Imaging type: CR (X-ray), CT, MR, US (Ultrasound) |
| `body_region` | Chest, Abdomen, Head, MSK, Brain, Breast, Thyroid |
| `finding` | What was found (or "Normal") |
| `ai_flagged` | Did the AI flag a finding? (True/False) |
| `ai_confidence` | AI's confidence score (0.0 to 1.0) |
| `radiologist_confirmed` | Did the radiologist confirm a real finding? (True/False) |

In [ ]:
# Load the CSV file from your forked repository
# If running in Colab, we load it directly from your GitHub fork.
# Replace YOUR-USERNAME with your actual GitHub username.

# ============================================================
# IMPORTANT: Replace YOUR-USERNAME below with your GitHub username
# ============================================================
GITHUB_USERNAME = 'YOUR-USERNAME'

url = f'https://raw.githubusercontent.com/{GITHUB_USERNAME}/research-starter/main/radiology_ai_findings.csv'
df = pd.read_csv(url)

print(f'Loaded {len(df)} studies from {GITHUB_USERNAME}\'s repository')
print(f'Columns: {list(df.columns)}')
print()
df.head(10)

## Step 2: Explore the Data

Before analyzing anything, we need to understand what we are working with. In Python, we use pandas to quickly summarize a dataset.

Key concepts:
- `df.shape` tells you (rows, columns)
- `df['column'].value_counts()` counts how many times each value appears
- `df.describe()` gives summary statistics for numeric columns

In [ ]:
# How many studies do we have?
print(f'Total studies: {df.shape[0]}')
print(f'Total columns: {df.shape[1]}')
print()

# How many studies per imaging modality?
print('Studies by modality:')
print(df['modality'].value_counts())
print()

# How many studies per body region?
print('Studies by body region:')
print(df['body_region'].value_counts())

In [ ]:
# Summary statistics for the AI confidence scores
print('AI Confidence Score Statistics:')
print(df['ai_confidence'].describe())
print()

# How many studies did the AI flag?
flagged = df['ai_flagged'].sum()
total = len(df)
print(f'AI flagged {flagged} out of {total} studies ({flagged/total*100:.1f}%)')

## Step 3: Visualize the Data

Charts help us see patterns that numbers alone might miss. We will create a bar chart comparing how many studies the AI flagged vs. how many the radiologist confirmed, broken down by imaging modality.

This is a fundamental visualization in AI validation research: **does the AI agree with the expert?**

In [ ]:
# Count AI-flagged and radiologist-confirmed studies by modality
modalities = df['modality'].unique()

ai_counts = df.groupby('modality')['ai_flagged'].sum()
rad_counts = df.groupby('modality')['radiologist_confirmed'].sum()

# Create a grouped bar chart
fig, ax = plt.subplots(figsize=(10, 6))

x = range(len(modalities))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], [ai_counts.get(m, 0) for m in modalities],
               width, label='AI Flagged', color='#0EA5E9')
bars2 = ax.bar([i + width/2 for i in x], [rad_counts.get(m, 0) for m in modalities],
               width, label='Radiologist Confirmed', color='#0B1D3A')

ax.set_xlabel('Imaging Modality', fontsize=12)
ax.set_ylabel('Number of Studies', fontsize=12)
ax.set_title('AI Flagged vs. Radiologist Confirmed Findings by Modality', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(modalities)
ax.legend()
ax.set_ylim(0, max(max(ai_counts), max(rad_counts)) + 3)

# Add count labels on top of each bar
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
            f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
            f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## Step 4: Calculate AI Accuracy Metrics

In radiology AI research, we evaluate models using these core metrics:

| Metric | Formula | What it tells you |
|--------|---------|-------------------|
| **Sensitivity** | TP / (TP + FN) | Of all real findings, how many did AI catch? |
| **Specificity** | TN / (TN + FP) | Of all normal studies, how many did AI correctly identify as normal? |
| **PPV** | TP / (TP + FP) | When AI flags something, how often is it real? |
| **NPV** | TN / (TN + FN) | When AI says normal, how often is it actually normal? |

Where:
- **TP** (True Positive) = AI flagged AND radiologist confirmed
- **FP** (False Positive) = AI flagged BUT radiologist did NOT confirm
- **FN** (False Negative) = AI did NOT flag BUT radiologist confirmed a finding
- **TN** (True Negative) = AI did NOT flag AND radiologist confirmed no finding

In [ ]:
# Calculate confusion matrix components
tp = len(df[(df['ai_flagged'] == True) & (df['radiologist_confirmed'] == True)])
fp = len(df[(df['ai_flagged'] == True) & (df['radiologist_confirmed'] == False)])
fn = len(df[(df['ai_flagged'] == False) & (df['radiologist_confirmed'] == True)])
tn = len(df[(df['ai_flagged'] == False) & (df['radiologist_confirmed'] == False)])

print('=== Overall AI Performance ===')
print(f'True Positives:  {tp}')
print(f'False Positives: {fp}')
print(f'False Negatives: {fn}')
print(f'True Negatives:  {tn}')
print()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0

print(f'Sensitivity: {sensitivity:.1%}')
print(f'Specificity: {specificity:.1%}')
print(f'PPV:         {ppv:.1%}')
print(f'NPV:         {npv:.1%}')

## Step 5: Performance by Modality

Overall accuracy can be misleading. An AI tool might perform well on chest X-rays but poorly on ultrasound. Let us break down performance by imaging modality.

In [ ]:
print('=== AI Sensitivity by Modality ===')
print('(Of real findings, what percentage did AI catch?)\n')

for modality in sorted(df['modality'].unique()):
    subset = df[df['modality'] == modality]
    mod_tp = len(subset[(subset['ai_flagged'] == True) & (subset['radiologist_confirmed'] == True)])
    mod_fn = len(subset[(subset['ai_flagged'] == False) & (subset['radiologist_confirmed'] == True)])
    mod_fp = len(subset[(subset['ai_flagged'] == True) & (subset['radiologist_confirmed'] == False)])
    mod_tn = len(subset[(subset['ai_flagged'] == False) & (subset['radiologist_confirmed'] == False)])

    sens = mod_tp / (mod_tp + mod_fn) if (mod_tp + mod_fn) > 0 else 0
    spec = mod_tn / (mod_tn + mod_fp) if (mod_tn + mod_fp) > 0 else 0

    print(f'{modality}:  Sensitivity = {sens:.1%}  |  Specificity = {spec:.1%}  '
          f'(TP={mod_tp}, FP={mod_fp}, FN={mod_fn}, TN={mod_tn})')

---

## Your Turn: Independent Analysis

Now it is your turn. Complete **all three tasks** below. Each task requires you to write or modify code. Your completed notebook will have unique outputs based on the choices you make.

### Task 1: Analyze a Body Region

Pick **one body region** from the dataset (Chest, Abdomen, Head, MSK, Brain, Breast, or Thyroid) and calculate the AI's sensitivity and specificity for that region only. Create a bar chart showing AI flagged vs. radiologist confirmed for that region's findings.

In [ ]:
# ============================================================
# TASK 1: Replace 'Chest' with your chosen body region
# ============================================================
MY_REGION = 'Chest'  # <-- Change this to your chosen region

region_df = df[df['body_region'] == MY_REGION]
print(f'\nAnalyzing: {MY_REGION}')
print(f'Total studies in {MY_REGION}: {len(region_df)}')
print()

# Calculate metrics for your chosen region
r_tp = len(region_df[(region_df['ai_flagged'] == True) & (region_df['radiologist_confirmed'] == True)])
r_fp = len(region_df[(region_df['ai_flagged'] == True) & (region_df['radiologist_confirmed'] == False)])
r_fn = len(region_df[(region_df['ai_flagged'] == False) & (region_df['radiologist_confirmed'] == True)])
r_tn = len(region_df[(region_df['ai_flagged'] == False) & (region_df['radiologist_confirmed'] == False)])

r_sens = r_tp / (r_tp + r_fn) if (r_tp + r_fn) > 0 else 0
r_spec = r_tn / (r_tn + r_fp) if (r_tn + r_fp) > 0 else 0

print(f'Sensitivity: {r_sens:.1%}')
print(f'Specificity: {r_spec:.1%}')
print(f'TP={r_tp}, FP={r_fp}, FN={r_fn}, TN={r_tn}')

# Create a bar chart for your region
findings = region_df['finding'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
findings.plot(kind='barh', ax=ax, color='#0D9488')
ax.set_xlabel('Number of Studies')
ax.set_title(f'Findings in {MY_REGION} Studies')
plt.tight_layout()
plt.show()

### Task 2: Confidence Threshold Analysis

The AI assigns a confidence score (0.0 to 1.0) to each study. A higher threshold means the AI is more selective about what it flags. Investigate: **what happens to sensitivity and specificity if we change the confidence threshold?**

Try at least 3 different thresholds and record your results.

In [ ]:
# ============================================================
# TASK 2: Try different confidence thresholds
# The code below tests threshold = 0.50. Add at least 2 more.
# ============================================================

thresholds = [0.50, 0.70, 0.85]  # <-- Add or change thresholds here

print('=== Confidence Threshold Analysis ===')
print(f'{"Threshold":<12} {"Sensitivity":<14} {"Specificity":<14} {"Flagged":<10}')
print('-' * 50)

for thresh in thresholds:
    # Re-flag studies using new threshold
    new_flagged = df['ai_confidence'] >= thresh

    t_tp = len(df[new_flagged & (df['radiologist_confirmed'] == True)])
    t_fp = len(df[new_flagged & (df['radiologist_confirmed'] == False)])
    t_fn = len(df[~new_flagged & (df['radiologist_confirmed'] == True)])
    t_tn = len(df[~new_flagged & (df['radiologist_confirmed'] == False)])

    t_sens = t_tp / (t_tp + t_fn) if (t_tp + t_fn) > 0 else 0
    t_spec = t_tn / (t_tn + t_fp) if (t_tn + t_fp) > 0 else 0
    total_flagged = new_flagged.sum()

    print(f'{thresh:<12.2f} {t_sens:<14.1%} {t_spec:<14.1%} {total_flagged:<10}')

print()
print('What do you notice about the trade-off between sensitivity and specificity?')

### Task 3: Write Your Interpretation

In the cell below, write 3-5 sentences answering these questions. This is graded for thoughtfulness, not length.

1. Which imaging modality did the AI perform **best** on? Which was **worst**? Why might that be?
2. Based on your threshold analysis, what confidence threshold would you recommend for clinical use, and why?
3. How does this relate to the **never-skilling** and **mis-skilling** risks from Lesson 1?

**Your interpretation:**

*Replace this text with your 3-5 sentence analysis. Be specific: reference the numbers you calculated above.*



---

## Save Your Work

Run the cell below to generate your unique completion record, then follow the instructions to save and commit.

### How to save and commit:
1. In Colab, go to **File > Save a copy in GitHub**
2. Select your forked `research-starter` repository
3. Keep the filename as `Research_1_AI_Data_Exploration.ipynb`
4. Write a commit message like: `Completed Research Module 1 analysis`
5. Click **OK** to commit

Your instructor will verify your work by checking your GitHub fork for:
- A commit with your completed notebook
- Unique outputs from your chosen body region and thresholds
- Your written interpretation in Task 3

In [ ]:
# Completion record — do not modify this cell
from datetime import datetime

print('=' * 50)
print('RESEARCH MODULE 1 — COMPLETION RECORD')
print('=' * 50)
print(f'GitHub Username:  {GITHUB_USERNAME}')
print(f'Completed:        {datetime.now().strftime("%Y-%m-%d %H:%M:%S UTC")}')
print(f'Region Analyzed:  {MY_REGION}')
print(f'Thresholds Tested: {thresholds}')
print(f'Dataset Size:     {len(df)} studies')
print('=' * 50)
print('Save this notebook to GitHub to submit your work.')